In [11]:
import xgboost as xgb
print(xgb.__version__)

3.0.4


In [12]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

c:\Users\saada\.conda\envs\ml_env\python.exe
3.0.4
c:\Users\saada\.conda\envs\ml_env\Lib\site-packages\xgboost\__init__.py


In [13]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

In [14]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv(r"D:\REGRESSION_ML_ENDTOEND\Regression_ML_EndtoEnd\data\processed\feature_engineered_train.csv")
eval_df  = pd.read_csv(r"D:\REGRESSION_ML_ENDTOEND\Regression_ML_EndtoEnd\data\processed\feature_engineered_eval.csv")


# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [15]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [ ]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder
# Use a relative path so it creates 'mlruns' right next to your notebook
# Use the file:/// prefix for local Windows paths
mlflow.set_tracking_uri(r"file:///D:/REGRESSION_ML_ENDTOEND/Regression_ML_EndtoEnd/mlruns")
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

c:\Users\saada\.conda\envs\ml_env\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[I 2026-03-23 23:54:34,178] A new study created in memory with name: no-name-bc5d3016-66b0-4b5f-a27a-cfde0a6c4c4a


[I 2026-03-23 23:56:16,508] Trial 0 finished with value: 75799.09134482684 and parameters: {'n_estimators': 958, 'max_depth': 6, 'learning_rate': 0.1953872574440415, 'subsample': 0.7384558556150846, 'colsample_bytree': 0.9658968577567861, 'min_child_weight': 7, 'gamma': 3.9266441678751023, 'reg_alpha': 0.0012961514861437257, 'reg_lambda': 0.2800858839334607}. Best is trial 0 with value: 75799.09134482684.
[I 2026-03-23 23:57:26,699] Trial 1 finished with value: 74683.6378999392 and parameters: {'n_estimators': 844, 'max_depth': 4, 'learning_rate': 0.15991545419400208, 'subsample': 0.5172058655963012, 'colsample_bytree': 0.5667817705360549, 'min_child_weight': 6, 'gamma': 0.16372914354368662, 'reg_alpha': 3.264321217683588e-06, 'reg_lambda': 1.1322903847891916e-06}. Best is trial 1 with value: 74683.6378999392.
[I 2026-03-23 23:58:00,121] Trial 2 finished with value: 74947.94288981966 and parameters: {'n_estimators': 465, 'max_depth': 4, 'learning_rate': 0.05280877279282254, 'subsample'

In [ ]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)


# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    
    # Use the sklearn flavor for better stability with XGBRegressor
    # Note: Use 'name' instead of 'artifact_path' as per the MLflow warning
    mlflow.sklearn.log_model(
        sk_model=best_model, 
        name="housing_xgboost_model"
    )

Final tuned model performance:
MAE: 30733.181228233985
RMSE: 68956.28115947364
R²: 0.9632542010332477


2026/03/23 23:30:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
